# Project 8 — Module 9: Fundamentos de Big Data
## Lesson 03 — Data Preparation

| | |
|---|---|
| **Author** | Jose Marcel Lopez Pino |
| **Framework** | CRISP-DM + LEAN |
| **Phase** | 03 — Data Preparation |
| **Module** | M9 — Fundamentos de Big Data (Alkemy Bootcamp) |
| **Dataset** | Brazilian E-Commerce (Olist) + Synthetic clickstream |
| **Date** | 2026-04 |

---

> **Executive Summary:**
> This notebook demonstrates RDD transformations and actions on RetailMax data.
> It creates RDDs and Pair RDDs from transaction data, applies transformations
> (map, filter, flatMap, distinct, sortBy), executes aggregation actions, documents
> the transformation lineage (DAG), and applies cache/persist strategy on reused
> datasets. The key output is a fully documented transformation pipeline with
> linaje explanation ready for Spark SQL processing in NB 04.

## Table of Contents

1. [Environment Setup — PySpark + Data Reload](#1-environment-setup)
2. [RDD Creation from DataFrames](#2-rdd-creation-from-dataframes)
3. [Transformations — map, filter, flatMap, distinct, sortBy](#3-transformations)
4. [Pair RDDs — Key-Value Operations](#4-pair-rdds)
5. [Actions — Aggregations (count, sum, mean, stdev)](#5-actions--aggregations)
6. [Cache/Persist Strategy](#6-cachepersist-strategy)
7. [Lineage and DAG Documentation](#7-lineage-and-dag-documentation)
8. [LEAN Filter — Waste Elimination Review](#8-lean-filter--waste-elimination-review)
9. [Decisions Log — Lesson 03](#9-decisions-log--lesson-03)
10. [Next Steps — Lesson 04 Preview](#10-next-steps--lesson-04-preview)

---

## 1. Environment Setup — PySpark + Data Reload <a id='1-environment-setup'></a>

Reload SparkSession and Olist DataFrames. RDDs are derived from DataFrames
to avoid the PySpark 4.1.1 Py4J/EOFException bug with `sc.textFile()`.

In [ ]:
# =============================================================================
# IMPORTS
# Standard Library
import os
import sys
import warnings
from pathlib import Path

# PySpark Python path (MUST be before PySpark imports)
os.environ['PYSPARK_PYTHON'] = sys.executable

# Core Data Science
import numpy as np
import pandas as pd

# Spark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    FloatType, DoubleType, TimestampType
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
warnings.formatwarning = lambda msg, *args, **kwargs: f'Warning: {msg}\n'
warnings.simplefilter('always')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Blues_d')

# Paths
DATA_RAW       = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')
DATA_FINAL     = Path('../data/final')

print('Imports ready.')

In [ ]:
# =============================================================================
# SPARK SESSION
# =============================================================================
spark = (
    SparkSession.builder
    .appName('RetailMax-BigData-Pipeline')
    .master('local[*]')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.ui.showConsoleProgress', 'false')
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel('WARN')

# Reload Olist DataFrames
olist_files = {
    'orders':       str(DATA_RAW / 'olist_orders_dataset.csv'),
    'order_items':  str(DATA_RAW / 'olist_order_items_dataset.csv'),
    'payments':     str(DATA_RAW / 'olist_order_payments_dataset.csv'),
    'reviews':      str(DATA_RAW / 'olist_order_reviews_dataset.csv'),
    'products':     str(DATA_RAW / 'olist_products_dataset.csv'),
    'customers':    str(DATA_RAW / 'olist_customers_dataset.csv'),
    'sellers':      str(DATA_RAW / 'olist_sellers_dataset.csv'),
    'geolocation':  str(DATA_RAW / 'olist_geolocation_dataset.csv'),
    'categories':   str(DATA_RAW / 'product_category_name_translation.csv'),
}

dfs = {}
for name, path in olist_files.items():
    dfs[name] = spark.read.csv(path, header=True, inferSchema=True, encoding='utf-8')

print(f'Spark {spark.version} | {len(dfs)} DataFrames loaded.')
print(f'Orders: {dfs["orders"].count():,} | Items: {dfs["order_items"].count():,}')

---

## 2. RDD Creation from DataFrames <a id='2-rdd-creation-from-dataframes'></a>

RDDs are the foundational abstraction in Spark — an immutable, distributed
collection of objects. We derive RDDs from DataFrames using `.rdd` to avoid
the Py4J/EOFException bug in PySpark 4.1.1.

**Key RDD properties:**
- **Immutable:** Once created, cannot be modified — transformations create new RDDs
- **Distributed:** Partitioned across cluster nodes (or local cores)
- **Lazy:** Transformations are recorded but not executed until an action triggers them
- **Typed:** Each element is a `Row` object (when derived from DataFrame)

In [ ]:
# =============================================================================
# CREATE RDDs FROM DATAFRAMES
# =============================================================================
# Core RDDs for transformation exercises
orders_rdd     = dfs['orders'].rdd
items_rdd      = dfs['order_items'].rdd
payments_rdd   = dfs['payments'].rdd
reviews_rdd    = dfs['reviews'].rdd
customers_rdd  = dfs['customers'].rdd

print('RDDs created from DataFrames:')
print(f'  orders_rdd     : {orders_rdd.getNumPartitions()} partitions')
print(f'  items_rdd      : {items_rdd.getNumPartitions()} partitions')
print(f'  payments_rdd   : {payments_rdd.getNumPartitions()} partitions')
print(f'  reviews_rdd    : {reviews_rdd.getNumPartitions()} partitions')
print(f'  customers_rdd  : {customers_rdd.getNumPartitions()} partitions')
print(f'\nRDD element type: {type(orders_rdd)}')

---

## 3. Transformations — map, filter, flatMap, distinct, sortBy <a id='3-transformations'></a>

Transformations are **lazy** — they define a computation plan but do not
execute until an action (like `count()` or `show()`) is called.

> **Analogy (ICI perspective):** Each transformation is a workstation in a
> production line. The DAG is the plant layout. Spark's optimizer eliminates
> redundant steps — just as Lean eliminates waste in a manufacturing flow.

**Note:** Due to PySpark 4.1.1 Py4J limitations, we demonstrate transformations
using the DataFrame API (which internally uses RDD transformations) and validate
results with `.show()` instead of `.collect()`.

In [ ]:
# =============================================================================
# TRANSFORMATION: map — extract total revenue per order item
# =============================================================================
# map equivalent: select + withColumn (creates new column from existing)
# Business logic: total_cost = price + freight_value

items_with_total = (
    dfs['order_items']
    .withColumn('total_cost', F.col('price') + F.col('freight_value'))
    .select('order_id', 'product_id', 'price', 'freight_value', 'total_cost')
)

print('MAP: Added total_cost = price + freight_value')
items_with_total.show(5, truncate=False)

In [ ]:
# =============================================================================
# TRANSFORMATION: filter — high-value orders only (total_cost > 200)
# =============================================================================
high_value_items = items_with_total.filter(F.col('total_cost') > 200)

total_items = dfs['order_items'].count()
high_value_count = high_value_items.count()
pct = (high_value_count / total_items) * 100

print(f'FILTER: total_cost > 200')
print(f'  Total items    : {total_items:>10,}')
print(f'  High-value     : {high_value_count:>10,} ({pct:.1f}%)')
print()
high_value_items.show(5, truncate=False)

In [ ]:
# =============================================================================
# TRANSFORMATION: flatMap — explode review words (unstructured data)
# =============================================================================
# flatMap equivalent: split review text into individual words
# This demonstrates working with unstructured data (consigna requirement)

review_words = (
    dfs['reviews']
    .filter(F.col('review_comment_message').isNotNull())
    .select(
        'review_id',
        F.explode(F.split(F.col('review_comment_message'), r'\s+')).alias('word')
    )
    .withColumn('word', F.lower(F.col('word')))
    .filter(F.length(F.col('word')) > 2)  # Remove very short words
)

total_words = review_words.count()
print(f'FLATMAP (explode): Split review text into individual words')
print(f'  Total words extracted: {total_words:,}')
print()
review_words.show(10, truncate=False)

In [ ]:
# =============================================================================
# TRANSFORMATION: distinct — unique values
# =============================================================================
# Unique order statuses
unique_statuses = dfs['orders'].select('order_status').distinct()
print('DISTINCT: Unique order statuses')
unique_statuses.show(truncate=False)

# Unique payment types
unique_payments = dfs['payments'].select('payment_type').distinct()
print('DISTINCT: Unique payment types')
unique_payments.show(truncate=False)

# Unique customer states
n_states = dfs['customers'].select('customer_state').distinct().count()
print(f'DISTINCT: Unique customer states: {n_states}')

In [ ]:
# =============================================================================
# TRANSFORMATION: sortBy — top 10 most expensive items
# =============================================================================
top_expensive = (
    items_with_total
    .orderBy(F.col('total_cost').desc())
    .limit(10)
)

print('SORTBY (orderBy): Top 10 most expensive order items')
top_expensive.show(10, truncate=False)

---

## 4. Pair RDDs — Key-Value Operations <a id='4-pair-rdds'></a>

Pair RDDs are RDDs of (key, value) tuples — essential for groupBy, reduceBy,
and join operations. In the DataFrame API, these are equivalent to
`groupBy().agg()` operations.

In [ ]:
# =============================================================================
# PAIR RDD EQUIVALENT: Revenue per order (groupBy + agg)
# =============================================================================
# Pair RDD concept: (order_id, total_revenue) — reduceByKey to sum
# DataFrame equivalent: groupBy('order_id').agg(sum('total_cost'))

revenue_per_order = (
    items_with_total
    .groupBy('order_id')
    .agg(
        F.sum('total_cost').alias('order_revenue'),
        F.count('product_id').alias('items_count'),
        F.avg('price').alias('avg_item_price')
    )
    .orderBy(F.col('order_revenue').desc())
)

print('PAIR RDD (groupBy+agg): Revenue per order — Top 10')
revenue_per_order.show(10, truncate=False)

In [ ]:
# =============================================================================
# PAIR RDD EQUIVALENT: Revenue per customer state
# =============================================================================
# Join orders + items + customers to get revenue by state
revenue_by_state = (
    dfs['orders']
    .join(dfs['order_items'], on='order_id', how='inner')
    .join(dfs['customers'], on='customer_id', how='inner')
    .withColumn('total_cost', F.col('price') + F.col('freight_value'))
    .groupBy('customer_state')
    .agg(
        F.sum('total_cost').alias('state_revenue'),
        F.countDistinct('order_id').alias('total_orders'),
        F.countDistinct('customer_id').alias('unique_customers')
    )
    .orderBy(F.col('state_revenue').desc())
)

print('PAIR RDD (multi-join + groupBy): Revenue by customer state')
revenue_by_state.show(10, truncate=False)

In [ ]:
# =============================================================================
# PAIR RDD EQUIVALENT: Word frequency from reviews (classic word count)
# =============================================================================
# This is the Spark equivalent of the classic MapReduce word count
# flatMap(split) → map(word, 1) → reduceByKey(+)

word_counts = (
    review_words
    .groupBy('word')
    .agg(F.count('*').alias('frequency'))
    .orderBy(F.col('frequency').desc())
)

print('PAIR RDD (word count): Top 20 most frequent words in reviews')
word_counts.show(20, truncate=False)

---

## 5. Actions — Aggregations (count, sum, mean, stdev) <a id='5-actions--aggregations'></a>

Actions **trigger execution** of the transformation chain. Unlike transformations
(lazy), actions return results to the driver program.

> **Analogy (ICI):** An action is the "pull signal" in a Kanban system —
> nothing moves until the downstream customer (action) requests it.

In [ ]:
# =============================================================================
# ACTIONS: Statistical aggregations on order items
# =============================================================================
stats = (
    items_with_total
    .agg(
        F.count('total_cost').alias('count'),
        F.sum('total_cost').alias('sum'),
        F.mean('total_cost').alias('mean'),
        F.stddev('total_cost').alias('stdev'),
        F.min('total_cost').alias('min'),
        F.max('total_cost').alias('max')
    )
)

print('ACTIONS: Statistical summary — total_cost (price + freight)')
stats.show(truncate=False)

# Collect result for display
row = stats.collect()[0]
print(f'Summary:')
print(f'  Count  : {row["count"]:>12,}')
print(f'  Sum    : {row["sum"]:>12,.2f}')
print(f'  Mean   : {row["mean"]:>12,.2f}')
print(f'  Stdev  : {row["stdev"]:>12,.2f}')
print(f'  Min    : {row["min"]:>12,.2f}')
print(f'  Max    : {row["max"]:>12,.2f}')

In [ ]:
# Check what review_score actually contains
dfs['reviews'].select('review_score').show(10, truncate=False)

In [ ]:
# Check all column types
dfs['reviews'].printSchema()

In [ ]:
# =============================================================================
# ACTIONS: Review score distribution
# =============================================================================
# Register as temp view and use SQL try_cast
dfs['reviews'].createOrReplaceTempView('reviews_raw')

reviews_clean = spark.sql("""
    SELECT *, try_cast(review_score AS INT) AS score_int
    FROM reviews_raw
    WHERE try_cast(review_score AS INT) IS NOT NULL
""")

review_stats = (
    reviews_clean
    .groupBy('score_int')
    .agg(F.count('*').alias('count'))
    .orderBy('score_int')
)

print('ACTIONS: Review score distribution')
review_stats.show(truncate=False)

# Overall review statistics
print('Overall review statistics:')
reviews_clean.agg(
    F.mean('score_int').alias('avg_score'),
    F.stddev('score_int').alias('stdev_score')
).show(truncate=False)

# How many rows were dropped?
original = dfs['reviews'].count()
clean = reviews_clean.count()
print(f'Original rows: {original:,} | Clean rows: {clean:,} | Dropped: {original - clean:,}')

---

## 6. Cache/Persist Strategy <a id='6-cachepersist-strategy'></a>

Caching stores intermediate results in memory to avoid recomputation.

> **Analogy (ICI):** Cache = buffer inventory at a bottleneck station.
> It costs memory (inventory holding cost) but prevents rework (recomputation).
> Cache only what is reused — unnecessary caching is waste (overproduction).

In [ ]:
# =============================================================================
# CACHE/PERSIST STRATEGY
# =============================================================================
from pyspark import StorageLevel

# Cache items_with_total — reused in Sections 3, 4, and 5
items_with_total.cache()
_ = items_with_total.count()  # Force materialization

# Cache revenue_per_order — will be reused in NB 04 for feature engineering
revenue_per_order.cache()
_ = revenue_per_order.count()

print('Cache Strategy:')
print(f'  items_with_total  : cached={items_with_total.is_cached}')
print(f'  revenue_per_order : cached={revenue_per_order.is_cached}')
print()
print('Storage levels available:')
print('  MEMORY_ONLY      — default .cache(), fast but lost on eviction')
print('  MEMORY_AND_DISK  — spills to disk if memory full')
print('  DISK_ONLY        — slow but handles very large datasets')
print()
print('Decision: MEMORY_ONLY (default) is appropriate for ~120 MB dataset.')
print('With 4g driver memory, eviction risk is negligible.')

---

## 7. Lineage and DAG Documentation <a id='7-lineage-and-dag-documentation'></a>

### What is Lineage?

Every RDD/DataFrame records its full transformation history (lineage). If a
partition is lost, Spark can recompute it from the original data by replaying
the transformations. This is Spark's fault-tolerance mechanism — no need for
data replication.

### What is a DAG?

The DAG (Directed Acyclic Graph) is the execution plan Spark builds from the
lineage. It groups transformations into **stages** (separated by shuffles)
and **tasks** (one per partition per stage).

> **Analogy (ICI):** The DAG is the process flow diagram of a manufacturing
> plant. Stages are workstations, tasks are individual work orders, and
> shuffles are material transfers between departments (expensive, minimize them).

In [ ]:
# =============================================================================
# LINEAGE: Explain plan for revenue_by_state
# =============================================================================
print('LINEAGE — Execution plan for revenue_by_state:')
print('=' * 70)
revenue_by_state.explain(mode='simple')

print()
print('LINEAGE — Extended plan (with optimizations):')
print('=' * 70)
revenue_by_state.explain(mode='extended')

### DAG Visualization — revenue_by_state Pipeline

```
Stage 1 (no shuffle — narrow transformations):
┌──────────────┐    ┌──────────────┐    ┌──────────────┐
│ orders CSV   │    │ items CSV    │    │ customers CSV│
│ (99K rows)   │    │ (113K rows)  │    │ (99K rows)   │
└──────┬───────┘    └──────┬───────┘    └──────┬───────┘
       │                   │                    │
       │            ┌──────┴───────┐            │
       │            │ withColumn   │            │
       │            │ (total_cost) │            │
       │            └──────┬───────┘            │
       │                   │                    │
       └───────────────────┼────────────────────┘
                          │
                   ┌──────┴───────┐
                   │    JOIN      │  ← shuffle (Stage boundary)
                   │ orders ⋈     │
                   │ items ⋈      │
                   │ customers    │
                   └──────┬───────┘
                          │
Stage 2 (after shuffle):
                   ┌──────┴───────┐
                   │  groupBy     │
                   │ (state)      │
                   │  + agg       │  ← shuffle (Stage boundary)
                   └──────┬───────┘
                          │
Stage 3 (final):
                   ┌──────┴───────┐
                   │   orderBy    │
                   │ (revenue ↓)  │  ← shuffle (Stage boundary)
                   └──────┬───────┘
                          │
                   ┌──────┴───────┐
                   │    show()    │  ← ACTION (triggers execution)
                   └──────────────┘
```

**Key observations:**
- **3 stages** separated by shuffles (joins and groupBy require data redistribution)
- **Narrow transformations** (withColumn, filter) stay within the same stage — no shuffle
- **Wide transformations** (join, groupBy, orderBy) trigger shuffles — expensive operations
- Spark's Catalyst optimizer may reorder or combine operations for efficiency

---

## 8. LEAN Filter — Waste Elimination Review <a id='8-lean-filter--waste-elimination-review'></a>

| LEAN Question | Answer | Action |
|---|---|---|
| Does every transformation link to a business metric? | ✅ map→revenue, filter→high-value, flatMap→review text, groupBy→state revenue | Proceed |
| Is there redundancy in transformations? | ✅ No — each demonstrates a different RDD concept required by consigna | Proceed |
| Is cache applied only to reused datasets? | ✅ items_with_total (3 sections), revenue_per_order (NB 04) — no unnecessary caching | Proceed |
| Could we skip the DAG documentation? | ⚠️ No — consigna explicitly requires DAG explanation | Keep — consigna compliance |

---

## 9. Decisions Log — Lesson 03 <a id='9-decisions-log--lesson-03'></a>

| # | Decision | Rationale | Alternatives Considered | LEAN Value? |
|---|----------|-----------|------------------------|-------------|
| D1 | Use DataFrame API for RDD operations | PySpark 4.1.1 Py4J/EOFException prevents direct RDD actions; DataFrame API uses same internal RDD engine | Direct RDD API (crashes), downgrade PySpark (breaks other projects) | ✅ |
| D2 | total_cost = price + freight_value | Business-relevant metric — represents actual customer spend | price only (ignores shipping cost impact) | ✅ |
| D3 | flatMap on review text (word frequency) | Demonstrates unstructured data processing — consigna requires structured + unstructured | Skip text analysis (misses consigna requirement) | ✅ |
| D4 | Cache items_with_total + revenue_per_order only | These are reused across multiple sections; other DataFrames are used once | Cache all DataFrames (wastes memory), no caching (recomputation waste) | ✅ |
| D5 | Document DAG with ASCII art + explain() | Visual + programmatic documentation covers both recruiter and technical reviewer audiences | explain() only (hard to read), screenshot only (not reproducible) | ✅ |

---

## 10. Next Steps — Lesson 04 Preview <a id='10-next-steps--lesson-04-preview'></a>

| Priority | Next Step | Target |
|---|---|---|
| 🔴 Immediate | Transform DataFrames with explicit schemas — apply StructType definitions | NB 04 |
| 🔴 Immediate | Execute Spark SQL queries for business metrics (revenue by category, top products) | NB 04 |
| 🔴 Immediate | Save processed results in Parquet format | NB 04 |
| 🟡 Short-term | Build MLlib pipeline — VectorAssembler + StringIndexer + LogReg + K-Means | NB 04 |
| 🟢 Long-term | Evaluate models and generate marketing insights report | NB 05 |

---

**← Previous:** [02 — Data Understanding](./02_data_understanding.ipynb)
**Next Phase →** [04 — Modeling](./04_modeling.ipynb)

In [ ]:
for field in dfs['order_items'].schema.fields:
    print(f'{field.name}: {field.dataType.simpleString()}')
print('---')
for field in dfs['products'].schema.fields:
    print(f'{field.name}: {field.dataType.simpleString()}')
print('---')
for field in dfs['payments'].schema.fields:
    print(f'{field.name}: {field.dataType.simpleString()}')
print('---')
for field in dfs['customers'].schema.fields:
    print(f'{field.name}: {field.dataType.simpleString()}')